In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix
from pathlib import Path

DATA_PATH = Path("news_data/cleaned_news_for_model.parquet")
df_model = pd.read_parquet(DATA_PATH)

print(DATA_PATH.resolve())
print(df_model.shape)
display(df_model.head(3))

/Users/stepan/code_projects/ml_projects/newsdata_analisis/news_data/cleaned_news_for_model.parquet
(146619, 10)


,source,url,archive_date,published_at,title,text,category_raw,collected_at,text_len_chars,text_len_words
0,lenta.ru,https://lenta.ru/news/2025/01/01/auto/,2025-01-01,2025-01-01 00:00:00,"Рост акцизов на топливо, увеличение утильсбора...",В России с 1 января 2025 года вступает в силу ...,Россия,2026-04-24T08:54:39+00:00,10241,1699
1,lenta.ru,https://lenta.ru/news/2025/01/01/v-rossii-stal...,2025-01-01,2025-01-01 00:02:00,В России стало дороже развестись,В России кратно увеличился размер госпошлины з...,Россия,2026-04-24T08:54:39+00:00,2361,335
2,lenta.ru,https://lenta.ru/news/2025/01/01/v-rossii-s-1-...,2025-01-01,2025-01-01 01:23:00,В России с 1 января повысили штрафы за нарушен...,В России с 1 января повысили штрафы за превыше...,Россия,2026-04-24T08:54:39+00:00,2940,472


In [2]:
df = df_model[["title", "text", "category_raw"]].copy()
df = df.dropna(subset=["text", "category_raw"])
df = df[df["text"].str.len() > 0]
df["category_raw"].value_counts()

category_raw
Мир                45383
Россия             41352
Экономика          32112
Наука и техника    10168
Спорт               9869
Культура            7735
Name: count, dtype: int64

In [3]:
X = df["text"]
y = df["category_raw"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    min_df=3
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [4]:
model = LinearSVC(
    max_iter=2000,
    class_weight="balanced"
)

model.fit(X_train_tfidf, y_train)
y_pred = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
weighted_f1 = f1_score(y_test, y_pred, average="weighted")

print("Accuracy:", accuracy)
print("Macro F1:", macro_f1)
print("Weighted F1:", weighted_f1)

Accuracy: 0.9482335288500887
Macro F1: 0.9569646999985176
Weighted F1: 0.9481728934932259


In [5]:
print(classification_report(y_test, y_pred))

                 precision    recall  f1-score   support

       Культура       0.96      0.97      0.97      1547
            Мир       0.95      0.96      0.96      9077
Наука и техника       0.96      0.95      0.96      2034
         Россия       0.95      0.92      0.93      8270
          Спорт       0.99      1.00      0.99      1974
      Экономика       0.93      0.94      0.93      6422

       accuracy                           0.95     29324
      macro avg       0.96      0.96      0.96     29324
   weighted avg       0.95      0.95      0.95     29324



In [6]:
labels = model.classes_
cm = confusion_matrix(y_test, y_pred, labels=labels)

cm_df = pd.DataFrame(
    cm,
    index=[f"true_{x}" for x in labels],
    columns=[f"pred_{x}" for x in labels]
)
cm_df

,pred_Культура,pred_Мир,pred_Наука и техника,pred_Россия,pred_Спорт,pred_Экономика
true_Культура,1505,15,0,21,1,5
true_Мир,6,8745,15,184,2,125
true_Наука и техника,1,45,1928,39,0,21
true_Россия,35,252,40,7622,8,313
true_Спорт,0,2,0,3,1968,1
true_Экономика,19,154,16,191,4,6038


In [7]:
cm_row_norm = cm / cm.sum(axis=1, keepdims=True)

cm_row_norm_df = pd.DataFrame(
    cm_row_norm,
    index=[f"true_{x}" for x in labels],
    columns=[f"pred_{x}" for x in labels]
)

cm_row_norm_df.round(3)

,pred_Культура,pred_Мир,pred_Наука и техника,pred_Россия,pred_Спорт,pred_Экономика
true_Культура,0.973,0.010,0.000,0.014,0.001,0.003
true_Мир,0.001,0.963,0.002,0.020,0.000,0.014
true_Наука и техника,0.000,0.022,0.948,0.019,0.000,0.010
true_Россия,0.004,0.030,0.005,0.922,0.001,0.038
true_Спорт,0.000,0.001,0.000,0.002,0.997,0.001
true_Экономика,0.003,0.024,0.002,0.030,0.001,0.940


In [8]:
error_rows = []

for i, true_label in enumerate(labels):
    for j, pred_label in enumerate(labels):
        if i != j:
            error_rows.append({
                "true": true_label,
                "pred": pred_label,
                "count": cm[i, j],
                "share_of_true_class": cm[i, j] / cm[i].sum(),
                "share_of_pred_class": cm[i, j] / cm[:, j].sum()
            })

error_analysis = pd.DataFrame(error_rows)
error_analysis.sort_values("count", ascending=False).head(30)

,true,pred,count,share_of_true_class,share_of_pred_class
19,Россия,Экономика,313,0.037848,0.048132
16,Россия,Мир,252,0.030472,0.027353
28,Экономика,Россия,191,0.029742,0.023697
7,Мир,Россия,184,0.020271,0.022829
26,Экономика,Мир,154,0.023980,0.016716
9,Мир,Экономика,125,0.013771,0.019222
11,Наука и техника,Мир,45,0.022124,0.004884
17,Россия,Наука и техника,40,0.004837,0.020010
12,Наука и техника,Россия,39,0.019174,0.004839
15,Россия,Культура,35,0.004232,0.022350
